# UGRIP-mediascope — MLP + XGBoost on LLM Signals

**Target**: 5-class factuality (`VERY LOW / LOW / MIXED / HIGH / VERY HIGH`)  
**Features**: (1) one-hot LLM verdicts (14-dim) · (2) sentence-transformer embedding of `full_response`  
**Data**: `bias_results/*.jsonl` merged with `snapshots.csv` ground truth

## 0 · Setup

In [ ]:
!pip install -q sentence-transformers xgboost scikit-learn

In [ ]:
import json, glob, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

SEED = 14
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1 · Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust to your actual Drive layout
BASE        = Path('/content/drive/MyDrive/UGRIP')
RESULTS_DIR = BASE / 'bias_results'
SNAPSHOTS   = BASE / 'snapshots.csv'

Mounted at /content/drive


## 2 · Load & merge

In [ ]:
FACT_LABELS  = ['VERY LOW', 'LOW', 'MIXED', 'HIGH', 'VERY HIGH']
BIAS_LABELS  = ['LEFT', 'LEFT-CENTER', 'LEAST BIASED', 'RIGHT-CENTER', 'RIGHT']
GENRE_LABELS = ['CONSPIRACY', 'PSEUDOSCIENCE', 'IMPOSTER', 'OTHER']

def load_joint_records(results_dir):
    records = []
    for path in glob.glob(str(results_dir / '*_joint_*.jsonl')):
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get('task') == 'joint':
                    records.append(rec)
    return records

records = load_joint_records(RESULTS_DIR)
print(f'Loaded {len(records)} joint records')

# Keep one record per outlet (last write wins if duplicates exist)
by_outlet = {}
for rec in records:
    by_outlet[rec['media_name']] = rec
records = list(by_outlet.values())
print(f'Unique outlets: {len(records)}')

Loaded 555 joint records
Unique outlets: 555


In [ ]:
snap = pd.read_csv(SNAPSHOTS)

rows = []
for rec in records:
    name = rec['media_name']
    match = snap[snap['media_name'] == name]
    if match.empty:
        continue
    gt_fact = match.iloc[0]['factuality'].strip().upper()
    if gt_fact not in FACT_LABELS:
        continue
    rows.append({
        'media_name':   name,
        'gt_factuality': gt_fact,
        'pred_fact':    (rec['verdicts'].get('factuality') or 'UNKNOWN').strip().upper(),
        'pred_bias':    (rec['verdicts'].get('bias')       or 'UNKNOWN').strip().upper(),
        'pred_genre':   (rec['verdicts'].get('genre')      or 'UNKNOWN').strip().upper(),
        'full_response': rec.get('full_response', ''),
    })

df = pd.DataFrame(rows)
print(df['gt_factuality'].value_counts())

gt_factuality
HIGH         200
MIXED        194
LOW           97
VERY LOW      53
VERY HIGH     11
Name: count, dtype: int64


## 3 · Feature engineering

In [ ]:
def one_hot(val, vocab):
    vec = [0] * len(vocab)
    if val in vocab:
        vec[vocab.index(val)] = 1
    return vec  # all-zeros when label is UNKNOWN / out-of-vocab

oh_feat = np.array([
    one_hot(r.pred_fact,  FACT_LABELS) +
    one_hot(r.pred_bias,  BIAS_LABELS) +
    one_hot(r.pred_genre, GENRE_LABELS)
    for r in df.itertuples()
], dtype=np.float32)

print(f'One-hot feature matrix: {oh_feat.shape}')  # (N, 14)

One-hot feature matrix: (555, 14)


In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')
texts   = df['full_response'].fillna('').tolist()

emb_feat = encoder.encode(texts, batch_size=64, show_progress_bar=True,
                          convert_to_numpy=True).astype(np.float32)
print(f'Embedding matrix: {emb_feat.shape}')  # (N, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding matrix: (555, 384)


In [ ]:
X = np.concatenate([oh_feat, emb_feat], axis=1)  # (N, 398)

le = LabelEncoder()
le.fit(FACT_LABELS)
y = le.transform(df['gt_factuality'].tolist())

print(f'X: {X.shape}  |  classes: {le.classes_}')

X: (555, 398)  |  classes: ['HIGH' 'LOW' 'MIXED' 'VERY HIGH' 'VERY LOW']


## 4 · Stratified train / val / test split (70 / 15 / 15)

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print(f'Train {len(y_train)} | Val {len(y_val)} | Test {len(y_test)}')

Train 388 | Val 83 | Test 84


## 5 · MLP (PyTorch)

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_mlp(X_tr, y_tr, X_v, y_v, n_classes, epochs=60, lr=1e-3):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model  = MLP(X_tr.shape[1], n_classes).to(device)

    weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
    crit    = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
    opt     = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    tr_loader = make_loader(X_tr, y_tr)

    best_val_f1, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_v).to(device)
            preds = model(xv).argmax(dim=1).cpu().numpy()
        val_f1 = f1_score(y_v, preds, average='macro', zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            print(f'  epoch {epoch:>3}  val macro-F1 = {val_f1:.4f}')

    model.load_state_dict(best_state)
    return model, device

In [ ]:
n_classes = len(le.classes_)
mlp_model, device = train_mlp(X_train, y_train, X_val, y_val, n_classes)

  epoch  10  val macro-F1 = 0.5583
  epoch  20  val macro-F1 = 0.6345
  epoch  30  val macro-F1 = 0.7260
  epoch  40  val macro-F1 = 0.6683
  epoch  50  val macro-F1 = 0.7047
  epoch  60  val macro-F1 = 0.6840


## 6 · XGBoost

In [ ]:
weights_tr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
sample_w   = np.array([weights_tr[c] for c in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=SEED,
    early_stopping_rounds=30,
)

xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_w,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

[0]	validation_0-mlogloss:1.56715


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [10:13:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[50]	validation_0-mlogloss:0.77342
[100]	validation_0-mlogloss:0.63445
[150]	validation_0-mlogloss:0.59178
[200]	validation_0-mlogloss:0.58150
[250]	validation_0-mlogloss:0.57572
[300]	validation_0-mlogloss:0.56893
[348]	validation_0-mlogloss:0.56871


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=30,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

## 7 · Evaluation

In [ ]:
def evaluate(name, y_true, y_pred, class_names):
    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n=== {name} ===')
    print(f'Accuracy  : {acc:.4f}')
    print(f'Macro-F1  : {macro_f1:.4f}')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    return {'model': name, 'accuracy': acc, 'macro_f1': macro_f1}

In [ ]:
mlp_model.eval()
with torch.no_grad():
    xt = torch.tensor(X_test).to(device)
    mlp_preds = mlp_model(xt).argmax(dim=1).cpu().numpy()

mlp_res = evaluate('MLP', y_test, mlp_preds, le.classes_)


=== MLP ===
Accuracy  : 0.7262
Macro-F1  : 0.5732
              precision    recall  f1-score   support

        HIGH       0.72      0.70      0.71        30
         LOW       0.87      0.87      0.87        15
       MIXED       0.74      0.77      0.75        30
   VERY HIGH       0.00      0.00      0.00         1
    VERY LOW       0.57      0.50      0.53         8

    accuracy                           0.73        84
   macro avg       0.58      0.57      0.57        84
weighted avg       0.73      0.73      0.73        84



In [ ]:
xgb_preds = xgb_model.predict(X_test)
xgb_res   = evaluate('XGBoost', y_test, xgb_preds, le.classes_)


=== XGBoost ===
Accuracy  : 0.8095
Macro-F1  : 0.6252
              precision    recall  f1-score   support

        HIGH       0.81      0.87      0.84        30
         LOW       0.81      0.87      0.84        15
       MIXED       0.83      0.83      0.83        30
   VERY HIGH       0.00      0.00      0.00         1
    VERY LOW       0.80      0.50      0.62         8

    accuracy                           0.81        84
   macro avg       0.65      0.61      0.63        84
weighted avg       0.81      0.81      0.81        84



In [ ]:
print('\n=== Summary ===')
pd.DataFrame([mlp_res, xgb_res]).set_index('model').round(4)


=== Summary ===


,accuracy,macro_f1
model,,
MLP,0.7262,0.5732
XGBoost,0.8095,0.6252


---
**Notes**

- One-hot encoding uses all-zeros for any verdict the model returned as UNKNOWN / out-of-vocabulary — this preserves the sample rather than discarding it.
- Class weights are recomputed on the training split only, so no label-distribution leakage from val/test occurs.
- The MLP checkpoint is selected by best val macro-F1, not final epoch, to guard against overfitting on the majority class.
- MIXED outlets are included as a 5th class; if you want the 4-class ablation simply filter `df` before building features.
- If the JSONL files contain results from multiple models, filter to a specific `model` value before building `by_outlet` to avoid mixing predictions.